# 03 - Predictive Model Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import pickle

## Load Data

In [ ]:
PROCESSED_DIR = r"C:\Users\Joshevan\Downloads\Deteksi-Banjir\modeling\data\processed"
CHECKPOINT_DIR = r"C:\Users\Joshevan\Downloads\Deteksi-Banjir\modeling\predictive_model\checkpoints"

daily_features = pd.read_csv(f"{PROCESSED_DIR}/daily_features.csv")
flood_targets = pd.read_csv(f"{PROCESSED_DIR}/flood_targets.csv")

with open(f"{CHECKPOINT_DIR}/evaluation_results.json") as f:
    eval_results = json.load(f)

with open(f"{CHECKPOINT_DIR}/model_meta.json") as f:
    model_meta = json.load(f)

print(f"daily_features: {daily_features.shape}")
print(f"flood_targets: {flood_targets.shape}")
print(f"Areas: {flood_targets['area_id'].nunique()}")

## Feature Importance

In [ ]:
with open(f"{CHECKPOINT_DIR}/classifier.pkl", "rb") as f:
    classifier = pickle.load(f)

feature_names = model_meta["feature_names"]
importances = classifier.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances,
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(importance_df["feature"], importance_df["importance"], color="steelblue")
ax.set_xlabel("Importance")
ax.set_title("Classifier Feature Importance")
plt.tight_layout()
plt.show()

## Per-Area Flood Risk Distribution

In [ ]:
risk_counts = (
    flood_targets
    .groupby(["area_id", "flood_risk"])
    .size()
    .unstack(fill_value=0)
)

risk_order = [c for c in ["normal", "waspada", "tergenang"] if c in risk_counts.columns]
risk_counts = risk_counts[risk_order]

fig, ax = plt.subplots(figsize=(10, 5))
risk_counts.plot(kind="bar", stacked=True, ax=ax, colormap="RdYlGn_r")
ax.set_xlabel("Area")
ax.set_ylabel("Days")
ax.set_title("Flood Risk Distribution per Area")
ax.legend(title="Risk Level")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Rainfall vs Flood Risk

In [ ]:
merged = daily_features.merge(
    flood_targets[["area_id", "date", "flood_risk"]],
    on=["area_id", "date"],
    how="inner",
)

risk_order = ["normal", "waspada", "tergenang"]
existing = [r for r in risk_order if r in merged["flood_risk"].values]

fig, ax = plt.subplots(figsize=(7, 5))
merged.boxplot(
    column="total_precipitation",
    by="flood_risk",
    ax=ax,
    positions=range(len(existing)),
)
ax.set_xticklabels(existing)
ax.set_xlabel("Flood Risk")
ax.set_ylabel("Total Precipitation (mm)")
ax.set_title("Total Precipitation by Flood Risk Level")
plt.suptitle("")
plt.tight_layout()
plt.show()

## Evaluation Metrics Summary

In [ ]:
rows = []
for metric, value in eval_results["classification"].items():
    rows.append({"Task": "Classification", "Metric": metric, "Value": value})
for metric, value in eval_results["regression"].items():
    rows.append({"Task": "Regression", "Metric": metric, "Value": value})

eval_df = pd.DataFrame(rows)
eval_df["Value"] = eval_df["Value"].map(lambda x: f"{x:.4f}" if isinstance(x, float) else x)
print(f"Test size: {eval_results['test_size']}")
print()
print(eval_df.to_string(index=False))